### EDA & CAPACITY ANALYSIS

#### Importing Libraries

In [86]:
import pandas as pd
import numpy as np
import plotly.express as px

print('Libraries imported successfully')

Libraries imported successfully


- To display maximum rows and columns for the dataset

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",100)

print("Maximum display option is chosen")

Maximum display option is chosen


#### Loading Cleaned Dataset

In [23]:
df = pd.read_csv("Cleaned_HHS_Care_Dataset.csv")

df.head()

,Date,CBP_Intake,CBP_Custody,HHS_Transfers,HHS_Care,HHS_Discharges,Days_Since_Previous_Report,Transfer_Exceeds_CBP,Discharge_Exceeds_HHS,Total_System_Load,Net_Intake,Care_Load_Growth_Rate,Rolling_7D_Load,Rolling_14D_Load,Cumulative_Net_Intake,Positive_Net_Intake,Load_Volatility_7_Obs,Discharge_Offset_Ratio,Anomaly_Flag
0,2023-01-12,33,53,34,6566,436,NaN,False,False,6619,-402,NaN,NaN,NaN,-402,False,NaN,12.823529,NaN
1,2023-01-22,32,49,39,7122,227,10.0,False,False,7171,-188,8.339628,NaN,NaN,-590,False,NaN,5.820513,NaN
2,2023-01-23,32,50,39,7280,181,1.0,False,False,7330,-142,2.217264,7040.00,NaN,-732,False,373.163503,4.641026,NaN
3,2023-01-24,47,42,47,7433,175,1.0,True,False,7475,-128,1.978172,7148.75,NaN,-860,False,374.353109,3.723404,Transfer exceedds reported CBP custody
4,2023-01-25,20,22,41,7538,180,1.0,True,False,7560,-139,1.137124,7231.00,NaN,-999,False,372.733819,4.390244,Transfer exceedds reported CBP custody


In [24]:
print('Shape:',df.shape)
print('Date Range: ',df['Date'].min()," to ",df['Date'].max())

Shape: (720, 19)
Date Range:  2023-01-12  to  2025-12-21


### Numerical Distribution
- Statistical Description

In [25]:
df[
    [
        'CBP_Intake',
        'CBP_Custody',
        'HHS_Transfers',
        'HHS_Care',
        'HHS_Discharges',
        'Total_System_Load',
        'Net_Intake'
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
CBP_Intake,720.0,93.523611,72.646625,0.0,12.00,99.0,147.25,333.0
CBP_Custody,720.0,171.494444,126.354965,7.0,36.00,193.0,263.25,531.0
HHS_Transfers,720.0,128.668056,97.322012,0.0,14.00,157.0,199.25,440.0
HHS_Care,720.0,6061.275000,2833.070109,1972.0,2467.75,6406.5,8010.25,11516.0
HHS_Discharges,720.0,173.406944,125.702841,0.0,19.75,181.0,267.00,505.0
Total_System_Load,720.0,6232.769444,2918.458078,2002.0,2500.75,6633.5,8249.50,11762.0
Net_Intake,720.0,-44.738889,95.863856,-465.0,-94.25,-11.5,5.25,206.0


### VISUALIZATION

#### Total System Load

In [35]:
fig = px.line(
    df,
    x = 'Date',
    y = 'Total_System_Load',
    title= 'UAC Total System Care Load Over Time',
    labels={
        'Date':'Date',
        'Total_System_Load' : 'Children Under Care'
    }
)

fig.update_layout(hovermode = 'x unified')

fig.show()

#### CBP VS HHS

In [36]:
fig = px.line(
    df,
    x = 'Date',
    y = ['CBP_Custody', 'HHS_Care'],
    title= 'CBP Custody vs HHS Care Load',
    labels={
        'Date' : 'Date',
        'value' : 'Children',
        'variable' : 'Care Stage'
    }
)

fig.update_layout(hovermode = 'x unified')

fig.show()

#### Net Intake

In [37]:
fig = px.line(
    df,
    x = 'Date',
    y = 'Net_Intake',
    title= 'Net Intake Pressure Over Time',
    labels={
        'Date' : 'Date',
        'Net_Intake' : 'Net Intake (Transfers - Discharges)'
    }
)

fig.add_hline(                               #horizontal zero line to show the distinction in data over time
    y=0,
    line_dash = 'dash'
)

fig.update_layout(hovermode = 'x unified')

fig.show()

#### Rolling Load

In [41]:
plot_df = df.rename(columns={
    'Total_System_Load': 'Observed Load',
    'Rolling_7D_Load': '7-Observation Rolling Mean',
    'Rolling_14D_Load': '14-Observation Rolling Mean'
})

fig = px.line(
    plot_df,
    x='Date',
    y=[
        'Observed Load',
        '7-Observation Rolling Mean',
        '14-Observation Rolling Mean'
    ],
    title='Observed and Smoothed System Care Load',
    labels={
        'Date': 'Date',
        'value': 'Children',
        'variable': 'Metric'
    }
)

fig.update_layout(
    hovermode='x unified'
)

fig.show()

#### Monthly Average System Load

In [47]:
monthly = (
    df.set_index('Date')
    .resample('ME')
    .agg({
        'Total_System_Load':'mean',
        'CBP_Custody':'mean',
        'HHS_Care':'mean',
        'HHS_Transfers':'sum',
        'HHS_Discharges': 'sum',
        'Net_Intake':'sum'
    }).reset_index()
)

monthly.head()

,Date,Total_System_Load,CBP_Custody,HHS_Care,HHS_Transfers,HHS_Discharges,Net_Intake
0,2023-01-31,7413.500000,43.875000,7369.625000,276,1856,-1580
1,2023-02-28,7967.105263,171.315789,7795.789474,2637,5183,-2546
2,2023-03-31,8190.473684,245.105263,7945.368421,3399,5375,-1976
3,2023-04-30,8092.857143,250.190476,7842.666667,3967,5668,-1701
4,2023-05-31,8404.090909,216.954545,8187.136364,3622,6256,-2634


In [48]:
fig = px.bar(
    monthly,
    x="Date",
    y="Total_System_Load",
    title="Average Monthly System Care Load",
    labels={
        "Date": "Month",
        "Total_System_Load": "Average Children Under Care"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

#### Monthly Net Intake

In [49]:
fig = px.bar(
    monthly,
    x="Date",
    y="Net_Intake",
    title="Monthly Net Intake Pressure",
    labels={
        "Date": "Month",
        "Net_Intake": "Cumulative Monthly Net Intake"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash",
    annotation_text="Balanced Flow"
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

#### Load Volatility

In [50]:
fig = px.line(
    df,
    x="Date",
    y="Load_Volatility_7_Obs",
    title="System Care Load Volatility",
    labels={
        "Date": "Date",
        "Load_Volatility_7_Obs": "Rolling Standard Deviation"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

#### Distribution of Net Intake

In [51]:
fig = px.histogram(
    df,
    x="Net_Intake",
    nbins=30,
    title="Distribution of Net Intake",
    labels={
        "Net_Intake": "Net Intake (Transfers − Discharges)"
    }
)

fig.add_vline(
    x=0,
    line_dash="dash",
    annotation_text="Zero"
)

fig.update_layout(
    height=500
)

fig.show()

#### Year-Wise Avg System Load

In [52]:
df["Year"] = df["Date"].dt.year

yearly = (
    df.groupby("Year")
      .agg(
          Average_System_Load=("Total_System_Load", "mean"),
          Average_CBP_Load=("CBP_Custody", "mean"),
          Average_HHS_Load=("HHS_Care", "mean"),
          Total_Transfers=("HHS_Transfers", "sum"),
          Total_Discharges=("HHS_Discharges", "sum"),
          Average_Net_Intake=("Net_Intake", "mean")
      )
      .reset_index()
)

yearly

,Year,Average_System_Load,Average_CBP_Load,Average_HHS_Load,Total_Transfers,Total_Discharges,Average_Net_Intake
0,2023,8846.543478,200.469565,8646.073913,36124,66244,-130.956522
1,2024,7319.864542,276.904382,7042.960159,52552,51689,3.438247
2,2025,2575.744770,32.907950,2542.836820,3965,6920,-12.364017


In [53]:
fig = px.bar(
    yearly,
    x="Year",
    y="Average_System_Load",
    title="Average System Care Load by Year",
    labels={
        "Year": "Year",
        "Average_System_Load": "Average Children Under Care"
    },
    text_auto=".0f"
)

fig.update_layout(
    height=500
)

fig.show()

#### Year-Wise CBP vs HHS

In [55]:
yearly_load = yearly.melt(
    id_vars="Year",
    value_vars=[
        "Average_CBP_Load",
        "Average_HHS_Load"
    ],
    var_name="Care Stage",
    value_name="Average Children"
)

yearly_load["Care Stage"] = yearly_load["Care Stage"].replace({
    "Average_CBP_Load": "CBP Custody",
    "Average_HHS_Load": "HHS Care"
})

yearly_load

,Year,Care Stage,Average Children
0,2023,CBP Custody,200.469565
1,2024,CBP Custody,276.904382
2,2025,CBP Custody,32.907950
3,2023,HHS Care,8646.073913
4,2024,HHS Care,7042.960159
5,2025,HHS Care,2542.836820


In [56]:
fig = px.bar(
    yearly_load,
    x="Year",
    y="Average Children",
    color="Care Stage",
    barmode="group",
    title="Average CBP vs HHS Care Load by Year"
)

fig.update_layout(
    height=500
)

fig.show()

#### Monthly HHS Transfers vs Discharges

In [59]:
monthly_flows = monthly.melt(
    id_vars="Date",
    value_vars=[
        "HHS_Transfers",
        "HHS_Discharges"
    ],
    var_name="Flow Type",
    value_name="Children"
)

monthly_flows["Flow Type"] = monthly_flows["Flow Type"].replace({
    "HHS_Transfers": "Transfers",
    "HHS_Discharges": "Discharges"
})

monthly_flows.head(5)

,Date,Flow Type,Children
0,2023-01-31,Transfers,276
1,2023-02-28,Transfers,2637
2,2023-03-31,Transfers,3399
3,2023-04-30,Transfers,3967
4,2023-05-31,Transfers,3622


In [60]:
fig = px.bar(
    monthly_flows,
    x="Date",
    y="Children",
    color="Flow Type",
    barmode="group",
    title="Monthly HHS Transfers vs Discharges",
    labels={
        "Date": "Month",
        "Children": "Total Children"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

### EDA Summary

In [61]:
eda_summary = pd.DataFrame({
    "Metric": [
        "Average HHS Care",
        "Maximum HHS Care",
        "Average HHS Transfers",
        "Average HHS Discharges",
        "Average Net Intake",
        "Positive Net Intake Observations",
        "Negative Net Intake Observations"
    ],
    "Value": [
        df["HHS_Care"].mean(),
        df["HHS_Care"].max(),
        df["HHS_Transfers"].mean(),
        df["HHS_Discharges"].mean(),
        df["Net_Intake"].mean(),
        (df["Net_Intake"] > 0).sum(),
        (df["Net_Intake"] < 0).sum()
    ]
})

eda_summary

,Metric,Value
0,Average HHS Care,6061.275000
1,Maximum HHS Care,11516.000000
2,Average HHS Transfers,128.668056
3,Average HHS Discharges,173.406944
4,Average Net Intake,-44.738889
5,Positive Net Intake Observations,238.000000
6,Negative Net Intake Observations,475.000000


- Major Discharge Period

In [62]:
top_discharge_dates = (
    df[
        [
            "Date",
            "HHS_Transfers",
            "HHS_Discharges",
            "Net_Intake",
            "HHS_Care"
        ]
    ]
    .sort_values("HHS_Discharges", ascending=False)
    .head(10)
)

top_discharge_dates

,Date,HHS_Transfers,HHS_Discharges,Net_Intake,HHS_Care
149,2023-08-31,237,505,-268,10140
225,2023-12-21,177,497,-320,11375
169,2023-10-01,152,493,-341,10598
145,2023-08-27,192,488,-296,9873
238,2024-01-11,11,476,-465,8829
215,2023-12-07,188,466,-278,10897
221,2023-12-17,125,456,-331,11092
183,2023-10-22,131,454,-323,9506
168,2023-09-28,193,447,-254,10984
144,2023-08-24,173,443,-270,10026


- Manjor negative intake period
- i.e. where Discharges exceed Transfers

In [63]:
top_negative_intake = (
    df[
        [
            "Date",
            "HHS_Transfers",
            "HHS_Discharges",
            "Net_Intake",
            "HHS_Care"
        ]
    ]
    .sort_values("Net_Intake")
    .head(10)
)

top_negative_intake

,Date,HHS_Transfers,HHS_Discharges,Net_Intake,HHS_Care
238,2024-01-11,11,476,-465,8829
0,2023-01-12,34,436,-402,6566
173,2023-10-05,53,427,-374,11130
178,2023-10-15,68,434,-366,10571
85,2023-05-25,64,408,-344,7546
169,2023-10-01,152,493,-341,10598
187,2023-10-26,52,390,-338,9235
221,2023-12-17,125,456,-331,11092
10,2023-02-05,15,342,-327,7586
183,2023-10-22,131,454,-323,9506


- Whether positive net intake tends to correspond with increases in HHS care, while negative net intake corresponds with decreases?

In [64]:
df["HHS_Care_Change"] = df["HHS_Care"].diff()

df[
    [
        "Date",
        "HHS_Care",
        "HHS_Transfers",
        "HHS_Discharges",
        "Net_Intake",
        "HHS_Care_Change"
    ]
].head(15)

,Date,HHS_Care,HHS_Transfers,HHS_Discharges,Net_Intake,HHS_Care_Change
0,2023-01-12,6566,34,436,-402,NaN
1,2023-01-22,7122,39,227,-188,556.0
2,2023-01-23,7280,39,181,-142,158.0
3,2023-01-24,7433,47,175,-128,153.0
4,2023-01-25,7538,41,180,-139,105.0
5,2023-01-29,7472,11,303,-292,-66.0
6,2023-01-30,7743,29,196,-167,271.0
7,2023-01-31,7803,36,158,-122,60.0
8,2023-02-01,7903,27,231,-204,100.0
9,2023-02-02,7879,23,298,-275,-24.0


In [65]:
# Actual Correlation

correlation = df[
    ["Net_Intake", "HHS_Care_Change"]
].corr()

correlation

,Net_Intake,HHS_Care_Change
Net_Intake,1.000000,0.333478
HHS_Care_Change,0.333478,1.000000


"Net intake exhibited a positive correlation of 0.33 with the change in HHS care load, indicating that inflow–outflow imbalance is associated with changes in active care load, although the moderate relationship suggests that additional operational and reporting factors influence observed care levels."

## FORECASTING

#### Let us look at the Reporting Frequency

In [69]:
df['Days_Between_Reports'] = df['Date'].diff().dt.days
df['Days_Between_Reports'].describe()

count    719.000000
mean       1.493741
std        1.001198
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       10.000000
Name: Days_Between_Reports, dtype: float64

In [70]:
df['Days_Between_Reports'].value_counts().sort_index().head(20)

Days_Between_Reports
1.0     558
2.0      10
3.0     122
4.0      23
5.0       3
6.0       1
7.0       1
10.0      1
Name: count, dtype: int64

In [72]:
print('No. of observations:',len(df))
print('Unique Reporting Dates:',df['Date'].nunique())
print('Date Range:',df['Date'].min().date(),'to',df['Date'].max().date())

No. of observations: 720
Unique Reporting Dates: 720
Date Range: 2023-01-12 to 2025-12-21


#### Weekly Forecasting Dataset

- Since the regular reporting dates are inconsistent
- Hence a weekly forecasting dataset need comes into the play

In [73]:
# new dataframe for weekly forecasting dataset

weekly = (
    df.set_index('Date').resample('W-SUN')
    .agg({
        'HHS_Care':'mean',
        'CBP_Custody':'mean',
        'HHS_Transfers':'sum',
        'HHS_Discharges':'sum',
        'Net_Intake':'sum',
        'Total_System_Load':'mean'
    }).reset_index()
)

- Understanding the Result

In [74]:
weekly.head(10)

,Date,HHS_Care,CBP_Custody,HHS_Transfers,HHS_Discharges,Net_Intake,Total_System_Load
0,2023-01-15,6566.000000,53.000000,34,436,-402,6619.00
1,2023-01-22,7122.000000,49.000000,39,227,-188,7171.00
2,2023-01-29,7430.750000,39.750000,138,839,-701,7470.50
3,2023-02-05,7782.800000,30.600000,130,1225,-1095,7813.40
4,2023-02-12,7766.400000,135.000000,535,1383,-848,7901.40
5,2023-02-19,7734.750000,224.000000,717,1118,-401,7958.75
6,2023-02-26,7857.000000,233.200000,985,1363,-378,8090.20
7,2023-03-05,7874.000000,210.000000,720,1090,-370,8084.00
8,2023-03-12,7843.333333,235.666667,469,890,-421,8079.00
9,2023-03-19,7833.200000,260.000000,965,1330,-365,8093.20


In [75]:
weekly.tail(10)

,Date,HHS_Care,CBP_Custody,HHS_Transfers,HHS_Discharges,Net_Intake,Total_System_Load
144,2025-10-19,2230.60,30.00,65,45,20,2260.60
145,2025-10-26,2259.40,36.40,74,56,18,2295.80
146,2025-11-02,2282.40,23.20,47,53,-6,2305.60
147,2025-11-09,2273.80,33.60,33,50,-17,2307.40
148,2025-11-16,2313.25,33.75,51,31,20,2347.00
149,2025-11-23,2348.60,41.60,47,40,7,2390.20
150,2025-11-30,2387.50,32.25,41,31,10,2419.75
151,2025-12-07,2416.60,31.00,38,49,-11,2447.60
152,2025-12-14,2444.20,38.60,34,39,-5,2482.80
153,2025-12-21,2475.00,39.00,52,56,-4,2514.00


In [76]:
weekly.shape

(154, 7)

- Checking for Missing Values

In [77]:
weekly.isnull().sum()

Date                 0
HHS_Care             0
CBP_Custody          0
HHS_Transfers        0
HHS_Discharges       0
Net_Intake           0
Total_System_Load    0
dtype: int64

In [78]:
weekly[weekly['HHS_Care'].isna()]

,Date,HHS_Care,CBP_Custody,HHS_Transfers,HHS_Discharges,Net_Intake,Total_System_Load


#### Visualizing the Forecast Dataset

In [79]:
fig = px.line(
    weekly,
    x="Date",
    y="HHS_Care",
    title="Weekly Average HHS Care Load",
    labels={
        "Date": "Week",
        "HHS_Care": "Average HHS Care Load"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

### TRAIN-TEST Dataset Splitting

In [81]:
split_index = int(len(weekly)*0.80)

train = weekly.iloc[:split_index].copy()
test = weekly.iloc[split_index:].copy()

print('Training Observations:',len(train))
print('Testing Observations:',len(test))

print('\nTraining Period:')
print(train['Date'].min().date(),'to',train['Date'].max().date())

print('\nTesting Period:')
print(test['Date'].min().date(),'to',test['Date'].max().date())

Training Observations: 123
Testing Observations: 31

Training Period:
2023-01-15 to 2025-05-18

Testing Period:
2025-05-25 to 2025-12-21


## Training Models & Evaluation

### A. Naive Forecasting Model

- Establishing the Baseline

In [82]:
test = test.copy()

test['Naive_Forecast'] = (
    weekly['HHS_Care']
    .shift(1)
    .iloc[test.index]
)

In [83]:
test[['Date','HHS_Care','Naive_Forecast']].head(10)

,Date,HHS_Care,Naive_Forecast
123,2025-05-25,2465.666667,2405.400000
124,2025-06-01,2509.250000,2465.666667
125,2025-06-08,2512.600000,2509.250000
126,2025-06-15,2513.500000,2512.600000
127,2025-06-22,2510.666667,2513.500000
128,2025-06-29,2462.800000,2510.666667
129,2025-07-06,2375.750000,2462.800000
130,2025-07-13,2272.000000,2375.750000
131,2025-07-20,2167.250000,2272.000000
132,2025-07-27,2106.600000,2167.250000


- Evaluation

In [88]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

#removing any possible missing values
baseline_eval = test.dropna(subset = ['HHS_Care', 'Naive_Forecast']).copy()

mae_naive = mean_absolute_error(
    baseline_eval['HHS_Care'],
    baseline_eval['Naive_Forecast']
)

rmse_eval = np.sqrt(
    mean_squared_error(
        baseline_eval['HHS_Care'],
        baseline_eval['Naive_Forecast']
    )
)

mape_naive = np.mean(
    np.abs(
        (baseline_eval['HHS_Care']-baseline_eval['Naive_Forecast'])/baseline_eval['HHS_Care']
    )
)*100

print(f'Naive MAE: {mae_naive:.2f}')
print(f'Naive RMSE: {rmse_eval:.2f}')
print(f'Naive MAPE: {mape_naive:.2f}%')

Naive MAE: 37.21
Naive RMSE: 45.16
Naive MAPE: 1.66%


- Visualization

In [89]:
fig = px.line(
    test,
    x="Date",
    y=["HHS_Care", "Naive_Forecast"],
    title="Naive Forecast vs Actual HHS Care Load",
    labels={
        "Date": "Week",
        "value": "HHS Care Load",
        "variable": "Series"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

### B. SARIMA Model

- Actual time-series model

In [90]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

- Preparing the Training Series

In [91]:
train_series = (
    train.set_index('Date')['HHS_Care'].asfreq('W-SUN')
)

In [92]:
sarima_model = SARIMAX(
    train_series,
    order=(1,1,1),
    seasonal_order=(1,0,0,4),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_result = sarima_model.fit(disp=False)

print(sarima_result.summary())

                                     SARIMAX Results                                      
Dep. Variable:                           HHS_Care   No. Observations:                  123
Model:             SARIMAX(1, 1, 1)x(1, 0, [], 4)   Log Likelihood                -810.324
Date:                            Sun, 23 Aug 2026   AIC                           1628.649
Time:                                    14:46:02   BIC                           1639.697
Sample:                                01-15-2023   HQIC                          1633.134
                                     - 05-18-2025                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.4926      0.084      5.870      0.000       0.328       0.657
ma.L1          0.4565      0.120   

- Generating the Test Forecast

In [94]:
sarima_forecast = sarima_result.get_forecast(steps = len(test))

test['SARIMA_Forecast'] = sarima_forecast.predicted_mean.values

#checking
test[
    [
        "Date",
        "HHS_Care",
        "Naive_Forecast",
        "SARIMA_Forecast"
    ]
].head(10)

,Date,HHS_Care,Naive_Forecast,SARIMA_Forecast
123,2025-05-25,2465.666667,2405.400000,2443.220009
124,2025-06-01,2509.250000,2465.666667,2461.835566
125,2025-06-08,2512.600000,2509.250000,2470.920580
126,2025-06-15,2513.500000,2512.600000,2475.301709
127,2025-06-22,2510.666667,2513.500000,2477.432350
128,2025-06-29,2462.800000,2510.666667,2478.481852
129,2025-07-06,2375.750000,2462.800000,2478.999011
130,2025-07-13,2272.000000,2375.750000,2479.253984
131,2025-07-20,2167.250000,2272.000000,2479.379644
132,2025-07-27,2106.600000,2167.250000,2479.441539


- Evaluation

In [95]:
mae_sarima = mean_absolute_error(
    test["HHS_Care"],
    test["SARIMA_Forecast"]
)

rmse_sarima = np.sqrt(
    mean_squared_error(
        test["HHS_Care"],
        test["SARIMA_Forecast"]
    )
)

mape_sarima = np.mean(
    np.abs(
        (test["HHS_Care"] - test["SARIMA_Forecast"])
        / test["HHS_Care"]
    )
) * 100

print(f"SARIMA MAE  : {mae_sarima:.2f}")
print(f"SARIMA RMSE : {rmse_sarima:.2f}")
print(f"SARIMA MAPE : {mape_sarima:.2f}%")

SARIMA MAE  : 229.75
SARIMA RMSE : 283.74
SARIMA MAPE : 10.82%


- Comparison & Visualization

In [96]:
fig = px.line(
    test,
    x="Date",
    y=[
        "HHS_Care",
        "Naive_Forecast",
        "SARIMA_Forecast"
    ],
    title="HHS Care Load: Actual vs Forecasts",
    labels={
        "Date": "Week",
        "value": "HHS Care Load",
        "variable": "Series"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=550
)

fig.show()

#### Comparing the Models: Naive Forecast vs SARIMA Forecast

In [98]:
results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "SARIMA"
    ],
    "MAE": [
        mae_naive,
        mae_sarima
    ],
    "RMSE": [
        rmse_eval,
        rmse_sarima
    ],
    "MAPE (%)": [
        mape_naive,
        mape_sarima
    ]
})

results.round(2)

,Model,MAE,RMSE,MAPE (%)
0,Naive Baseline,37.21,45.16,1.66
1,SARIMA,229.75,283.74,10.82


"A persistence-based naive forecasting model outperformed the tested SARIMA specification, indicating strong short-term persistence in the weekly HHS care-load series."

#### C. XGBOOST
- So that our ML Model can potentially learn relationships that Naive Bayes' Forecast was unable to establish and predict

In [100]:
from xgboost import XGBRegressor

- creating Lag Features

In [101]:
ml_df = weekly.copy()

# HHS Care lags
ml_df["HHS_Care_Lag_1"] = ml_df["HHS_Care"].shift(1)
ml_df["HHS_Care_Lag_2"] = ml_df["HHS_Care"].shift(2)
ml_df["HHS_Care_Lag_4"] = ml_df["HHS_Care"].shift(4)
ml_df["HHS_Care_Lag_8"] = ml_df["HHS_Care"].shift(8)

# Flow lags
ml_df["Transfers_Lag_1"] = ml_df["HHS_Transfers"].shift(1)
ml_df["Discharges_Lag_1"] = ml_df["HHS_Discharges"].shift(1)
ml_df["Net_Intake_Lag_1"] = ml_df["Net_Intake"].shift(1)

# Rolling features
ml_df["HHS_Care_Rolling_4"] = (
    ml_df["HHS_Care"]
    .shift(1)
    .rolling(4)
    .mean()
)

ml_df["HHS_Care_Rolling_8"] = (
    ml_df["HHS_Care"]
    .shift(1)
    .rolling(8)
    .mean()
)

ml_df["HHS_Care_Rolling_8_STD"] = (
    ml_df["HHS_Care"]
    .shift(1)
    .rolling(8)
    .std()
)

- Adding Calendar features

In [102]:
ml_df["Month"] = ml_df["Date"].dt.month
ml_df["Quarter"] = ml_df["Date"].dt.quarter
ml_df["Year"] = ml_df["Date"].dt.year

In [103]:
ml_df = ml_df.dropna().reset_index(drop=True)

In [104]:
ml_df.shape

(146, 20)

- Train/Test Splitting

In [105]:
split_index = int(len(ml_df) * 0.80)

ml_train = ml_df.iloc[:split_index].copy()
ml_test = ml_df.iloc[split_index:].copy()

print("ML Training:", len(ml_train))
print("ML Testing :", len(ml_test))

print("\nTraining:")
print(ml_train["Date"].min(), "to", ml_train["Date"].max())

print("\nTesting:")
print(ml_test["Date"].min(), "to", ml_test["Date"].max())

ML Training: 116
ML Testing : 30

Training:
2023-03-12 00:00:00 to 2025-05-25 00:00:00

Testing:
2025-06-01 00:00:00 to 2025-12-21 00:00:00


- Features & Target

In [106]:
features = [
    "HHS_Care_Lag_1",
    "HHS_Care_Lag_2",
    "HHS_Care_Lag_4",
    "HHS_Care_Lag_8",
    "Transfers_Lag_1",
    "Discharges_Lag_1",
    "Net_Intake_Lag_1",
    "HHS_Care_Rolling_4",
    "HHS_Care_Rolling_8",
    "HHS_Care_Rolling_8_STD",
    "Month",
    "Quarter",
    "Year"
]

target = "HHS_Care"

X_train = ml_train[features]
y_train = ml_train[target]

X_test = ml_test[features]
y_test = ml_test[target]

- Training

In [107]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


- Forecasting

In [109]:
ml_test["XGBoost_Forecast"] = xgb_model.predict(X_test)

ml_test[
    [
        "Date",
        "HHS_Care",
        "XGBoost_Forecast"
    ]
].head(10)

,Date,HHS_Care,XGBoost_Forecast
116,2025-06-01,2509.250000,2412.223145
117,2025-06-08,2512.600000,2407.170410
118,2025-06-15,2513.500000,2391.656006
119,2025-06-22,2510.666667,2407.170410
120,2025-06-29,2462.800000,2394.671875
121,2025-07-06,2375.750000,2419.649414
122,2025-07-13,2272.000000,2391.481689
123,2025-07-20,2167.250000,2365.765869
124,2025-07-27,2106.600000,2321.096680
125,2025-08-03,2064.400000,2339.796875


- Evaluation

In [110]:
mae_xgb = mean_absolute_error(
    y_test,
    ml_test["XGBoost_Forecast"]
)

rmse_xgb = np.sqrt(
    mean_squared_error(
        y_test,
        ml_test["XGBoost_Forecast"]
    )
)

mape_xgb = np.mean(
    np.abs(
        (y_test - ml_test["XGBoost_Forecast"])
        / y_test
    )
) * 100

print(f"XGBoost MAE  : {mae_xgb:.2f}")
print(f"XGBoost RMSE : {rmse_xgb:.2f}")
print(f"XGBoost MAPE : {mape_xgb:.2f}%")

XGBoost MAE  : 192.99
XGBoost RMSE : 231.10
XGBoost MAPE : 9.04%


- Visualization

In [114]:
fig = px.line(
    ml_test,
    x="Date",
    y=["HHS_Care", "XGBoost_Forecast"],
    title="XGBoost Forecast vs Actual HHS Care Load",
    labels={
        "Date": "Week",
        "value": "HHS Care Load",
        "variable": "Series"
    }
)

fig.update_layout(
    hovermode="x unified",
    height=500
)

fig.show()

#### Comparing the Models: Naive Forecast vs SARIMA vs XGBoost

In [112]:
final_results = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "SARIMA",
        "XGBoost"
    ],
    "MAE": [
        mae_naive,
        mae_sarima,
        mae_xgb
    ],
    "RMSE": [
        rmse_eval,
        rmse_sarima,
        rmse_xgb
    ],
    "MAPE (%)": [
        mape_naive,
        mape_sarima,
        mape_xgb
    ]
})

final_results.round(2)

,Model,MAE,RMSE,MAPE (%)
0,Naive Baseline,37.21,45.16,1.66
1,SARIMA,229.75,283.74,10.82
2,XGBoost,192.99,231.10,9.04


"Model complexity did not translate into superior forecasting performance. The Naive Baseline substantially outperformed both SARIMA and XGBoost, indicating strong temporal persistence in the target variable and suggesting that more complex models may provide limited additional predictive value for this dataset."